In [5]:
pip install opencv-python pillow pytesseract easyocr requests numpy langdetect groq google-generativeai

Note: you may need to restart the kernel to use updated packages.


  Attempting uninstall: googleapis-common-protos
    Found existing installation: googleapis-common-protos 1.54.0
    Uninstalling googleapis-common-protos-1.54.0:
      Successfully uninstalled googleapis-common-protos-1.54.0


In [24]:
import os
import cv2
import numpy as np
import requests
from PIL import Image
import pytesseract
import easyocr
import json
from io import BytesIO
from groq import Groq
import google.generativeai as genai
from langdetect import detect
import re

GROQ_API_KEY = "YOUR_GROQ_API_KEY"  # Replace with your API key
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"  # Replace with your API key

groq_client = Groq(api_key=GROQ_API_KEY)
genai.configure(api_key=GOOGLE_API_KEY)

reader_en = easyocr.Reader(['en'], gpu=False)  
reader_hi = easyocr.Reader(['hi', 'en'], gpu=False) 

def load_image(image_path_or_url):
    """Load image from local path or URL."""
    try:
        if image_path_or_url.startswith(('http://', 'https://')):
            response = requests.get(image_path_or_url)
            img = Image.open(BytesIO(response.content))
            return np.array(img)
        else:
            return cv2.imread(image_path_or_url)
    except Exception as e:
        print(f"Error loading image: {e}")
        return None

def preprocess_image(image, is_hindi=False):
    """Preprocess the image for better OCR results with language-specific adjustments."""
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    
    if is_hindi:
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        denoised = cv2.fastNlMeansDenoising(thresh, None, 5, 7, 21)
        if max(denoised.shape) > 3000:
            scale_factor = 3000 / max(denoised.shape)
            width = int(denoised.shape[1] * scale_factor)
            height = int(denoised.shape[0] * scale_factor)
            denoised = cv2.resize(denoised, (width, height))
        
        return denoised
    else:
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 11, 2)
        
       
        denoised = cv2.fastNlMeansDenoising(thresh, None, 10, 7, 21)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(denoised)
        if max(enhanced.shape) > 3000:
            scale_factor = 3000 / max(enhanced.shape)
            width = int(enhanced.shape[1] * scale_factor)
            height = int(enhanced.shape[0] * scale_factor)
            enhanced = cv2.resize(enhanced, (width, height))
        
        return enhanced

def detect_language(text):
    """Improved language detection with a focus on Hindi characters."""
    hindi_pattern = re.compile(r'[\u0900-\u097F]')
    if hindi_pattern.search(text):
        return 'hi'
    
    
    try:
        if len(text.strip()) < 10:
            return "en"  
            
        lang = detect(text)
        return lang if lang == 'hi' else 'en'
    except:
        return "en" 
def extract_text_with_tesseract(image, language='eng'):
    """Extract text using Tesseract OCR with improved configuration."""
    
    lang_map = {
        'en': 'eng',
        'hi': 'hin'
    }
    
    tesseract_lang = lang_map.get(language, 'eng')
    
    try:
       
        if language == 'hi':
            config = '--psm 6 --oem 3'  
        else:
            config = '--psm 4 --oem 3'  
        
        text = pytesseract.image_to_string(
            image, 
            lang=tesseract_lang,
            config=config
        )
        return text
    except Exception as e:
        print(f"Tesseract error: {e}")
        return ""

def extract_text_with_easyocr(image, language='en'):
    """Extract text using EasyOCR with improved settings for Hindi."""
    try:
        if language == 'hi':
            
            results = reader_hi.readtext(
                image, 
                detail=0, 
                paragraph=True,
                decoder='greedy',
                beamWidth=5,
                batch_size=4,
                contrast_ths=0.1,
                adjust_contrast=0.5,
                text_threshold=0.6
            )
        else: 
            results = reader_en.readtext(
                image, 
                detail=0, 
                paragraph=True
            )
        
        return "\n".join(results)
    except Exception as e:
        print(f"EasyOCR error: {e}")
        return ""

def clean_hindi_menu_text(text):
    """Clean up common OCR errors in Hindi menu text."""
    cleaned = text.replace('।', '.').replace('₹', 'Rs.')
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    cleaned = re.sub(r'(\d+)/-', r'\1', cleaned)
    
    return cleaned

def translate_text(text, source_lang='hi', target_lang='en'):
    """Enhanced translation for menu text."""
    if source_lang == target_lang:
        return text
    
    try:
        model = genai.GenerativeModel('gemini-pro')
        response = model.generate_content(
            f"""Translate the following menu text from Hindi to English:
            
            {text}
            
            Important guidelines:
            1. Preserve all food item names as closely as possible
            2. Keep all prices and numbers exactly as they appear
            3. Maintain the menu structure and categories
            4. For dishes with no direct translation, transliterate the name and provide a brief description if possible
            5. Format the output to preserve menu sections and item-price associations
            """
        )
        return response.text
    except Exception as e:
        print(f"Translation error: {e}")
        return text

def extract_prices_and_items(text):
    """Pre-process text to better extract menu items and prices with improved regex patterns."""
    items_with_prices = []
    lines = text.split('\n')
    
    for line in lines:
        if not line.strip() or len(line.strip()) < 3:
            continue
            
        price_match = re.search(r'(?:₹|Rs\.|Rs|रु|रू)?\.?\s*(\d+(?:\.\d+)?)(?:/-|/|-|$|\s)', line)
        
        if price_match:
            price = price_match.group(1)
       
            item_parts = re.split(r'(?:₹|Rs\.|Rs|रु|रू)?\.?\s*\d+(?:\.\d+)?(?:/-|/|-|$|\s)', line)
            
            if item_parts and item_parts[0].strip():
                item_name = re.sub(r'\.{2,}', ' ', item_parts[0]).strip()
                item_name = re.sub(r'\s{2,}', ' ', item_name)
                
                items_with_prices.append({
                    "item": item_name,
                    "price": price
                })
    
    return items_with_prices

def structure_menu_with_llm(text, extracted_items, model_choice="groq"):
    """Structure the extracted menu text into JSON using an LLM with improved prompt."""
    items_text = "\n".join([f"Item: {item['item']}, Price: {item['price']}" for item in extracted_items]) if extracted_items else "No items pre-extracted"
    
    prompt = f"""
    Parse the following menu text into a structured JSON format:
    
    MENU TEXT:
    {text}
    
    PRE-EXTRACTED ITEMS AND PRICES:
    {items_text}
    
    Extract the restaurant name from the menu, and organize menu items by category.
    The output should be a valid JSON with this structure:
    {{
      "restaurant_name": "Name of restaurant (or 'Unknown' if not found)",
      "menu": [
        {{"item": "Menu item name", "price": "Price with currency symbol", "category": "Appropriate category"}}
      ]
    }}
    
    Instructions:
    1. For each menu item, include both the item name and its price
    2. Categorize items appropriately (e.g., Appetizers, Main Course, Desserts, Beverages, etc.)
    3. If a price is not specified for an item, use "N/A"
    4. Look for category headers in the menu text (like "SPECIAL THALI", "MAIN COURSE", etc.)
    5. If a currency symbol is missing, add the appropriate one (e.g., "₹")
    6. Menu items with prices are likely actual menu items
    7. If item names are transliterated from Hindi, correct any obvious errors
    
    Return ONLY the JSON in your response, no additional text.
    """
    
    try:
        if model_choice.lower() == "groq":
            response = groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama3-70b-8192"
            )
            return response.choices[0].message.content
        else:  
            model = genai.GenerativeModel('gemini-pro')
            response = model.generate_content(prompt)
            return response.text
    except Exception as e:
        print(f"LLM processing error: {e}")
        
        if extracted_items:
            return json.dumps({
                "restaurant_name": "Unknown",
                "menu": extracted_items
            }, indent=2)
        else:
            return json.dumps({
                "restaurant_name": "Unknown",
                "menu": []
            }, indent=2)

def process_menu_image(image_path_or_url, ocr_engine="easyocr", llm="groq", force_language=None):
    """Complete pipeline to process a menu image and return structured JSON."""
   
    print(f"Loading image from: {image_path_or_url}")
    image = load_image(image_path_or_url)
    if image is None:
        return json.dumps({"error": "Failed to load image"})
    
   
    print("Performing initial text extraction for language detection...")
    initial_preprocessing = preprocess_image(image, is_hindi=False)
    
    if ocr_engine.lower() == "tesseract":
        initial_text = extract_text_with_tesseract(initial_preprocessing)
    else:
        initial_text = extract_text_with_easyocr(initial_preprocessing, 'en')
    
    
    if force_language:
        detected_lang = force_language
        print(f"Using forced language: {detected_lang}")
    else:
        detected_lang = detect_language(initial_text)
        print(f"Detected language: {detected_lang}")
    
  
    print(f"Preprocessing image for {detected_lang}...")
    preprocessed = preprocess_image(image, is_hindi=(detected_lang == 'hi'))
    
    
    debug_img_path = f"preprocessed_menu_{detected_lang}.jpg"
    cv2.imwrite(debug_img_path, preprocessed)
    print(f"Saved preprocessed image to {debug_img_path}")
    
 
    print(f"Extracting text with {ocr_engine} and language {detected_lang}...")
    if ocr_engine.lower() == "tesseract":
        text = extract_text_with_tesseract(preprocessed, detected_lang)
    else:
        text = extract_text_with_easyocr(preprocessed, detected_lang)
    
    if not text:
        return json.dumps({"error": "Failed to extract any text from image"})
    
    
    if detected_lang == 'hi':
        text = clean_hindi_menu_text(text)
    
   
    print("Extracted text:")
    print(text)
    print("-" * 50)
    
   
    if detected_lang == 'hi':
        print("Translating from Hindi to English...")
        translated_text = translate_text(text, detected_lang, 'en')
        print("Translated text:")
        print(translated_text)
        print("-" * 50)
        
    
        text_for_processing = translated_text
    else:
        text_for_processing = text
    
    
    print("Pre-extracting menu items and prices...")
    extracted_items = extract_prices_and_items(text_for_processing)
    print(f"Pre-extracted {len(extracted_items)} items with prices")
    
 
    print(f"Structuring menu with {llm}...")
    structured_menu = structure_menu_with_llm(text_for_processing, extracted_items, llm)
    
    
    try:
        json_menu = json.loads(structured_menu)
      
        if "menu" in json_menu and isinstance(json_menu["menu"], list):
            for item in json_menu["menu"]:
              
                if "price" in item and re.match(r'^\d+(?:\.\d+)?$', str(item["price"])):
                    
                    currency_symbol = "₹" if detected_lang == 'hi' else "$"
                    item["price"] = currency_symbol + str(item["price"])
        
       
        json_menu["metadata"] = {
            "detected_language": detected_lang,
            "ocr_engine": ocr_engine,
            "llm_engine": llm,
            "extracted_items_count": len(extracted_items)
        }
        
        return json.dumps(json_menu, indent=2, ensure_ascii=False)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        fallback = {
            "error": "Failed to structure menu into valid JSON, using fallback",
            "restaurant_name": "Unknown",
            "detected_language": detected_lang,
            "menu": extracted_items
        }
        return json.dumps(fallback, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    
    result = process_menu_image(
        r"PATH_OF_IMAGE_FROM_LOCAL_DISK",  # Replace with your image path
        ocr_engine="easyocr",
        llm="groq",
        force_language="hi"  
    )
    
   
    print("\nFINAL RESULT:")
    print(result)
    
    with open("structured_menu.json", "w", encoding="utf-8") as f:
        f.write(result)

Using CPU. Note: This module is much faster with a GPU.
Using CPU. Note: This module is much faster with a GPU.


Loading image from: C:\Users\shivanshi garg\OneDrive\Desktop\canva-orange-and-black-bold-geometric-restaurant-menu-AX4bhelWqNA.jpg
Performing initial text extraction for language detection...
Using forced language: hi
Preprocessing image for hi...
Saved preprocessed image to preprocessed_menu_hi.jpg
Extracting text with easyocr and language hi...
Extracted text:
=00D MENU Paucek and Lage Restaurant MAIN COURSE Cheeseburger Cheese sandwich Chicken burgers Spicy chicken Hot dog $34 $22 $23 $33 $24 APPETIZERS Fruit Salad Cocktails Nuggets Sandwich French Fries $13 $12 $14 $13 $15 BEVERAGES Milk Shake Iced Tea Orange Juice Lemon Tea Coffee $3 $२ $4 $3 $5 123-456-7890 १२३ Anywhere St , Any City
--------------------------------------------------
Translating from Hindi to English...
Translation error: 404 models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
Translated te